# FitzHugh–Nagumo Excitable Medium Conditioning

This notebook reproduces the FitzHugh–Nagumo experiment of *LatentFlow: A General Framework for Conditioning Stochastic Processes* (Sharrock, Astfalck & Moss, 2026; Figures 5 and 18). An excitable medium follows the stochastic FitzHugh–Nagumo equations

$$\partial_t u = D\,\Delta u + u - \tfrac{1}{3}u^3 - v + I(\mathbf{x}) + \sigma(\mathbf{x})\,\xi(\mathbf{x}, t),\qquad \partial_t v = \varepsilon\,(u + a - b v),$$

where $u$ is the activator, $v$ the recovery variable and $\xi$ space–time white noise. Two **pacemaker** patches (lower-left and upper-right) receive a sub-threshold drive $I$ and extra noise, so in the prior each fires only sometimes, launching an activation front. The noise on a $32 \times 32$ grid at 40 times gives about **41K** Gaussian innovations. Guidance then conditions on the firing pattern:

1. **source 1 only**: the lower-left pacemaker fires early ($t \le 1.3$) while the upper-right stays silent;
2. **source 2 only**: the upper-right fires late ($t \ge 1.8$, silent before) while the lower-left stays silent;
3. **both fire**, in sequence: lower-left early, then upper-right late.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.patches import Rectangle

from flow_inference.conditions import BinaryCondition, EqualityCondition
from flow_inference.flows import GaussianIncrementFlow
from flow_inference.guidance import GuidanceMC
from flow_inference.maps import BaseMap
from flow_inference.sampler import IntegratorSampler

torch.set_default_dtype(torch.float64)

## 1. Configuration

Grid units for space; the solver takes four Euler substeps per noise step for stability.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
torch.manual_seed(SEED)

G = 32  # grid size
NUM_FRAMES = 40  # noise steps / returned frames
T_FINAL = 4.0
DT = T_FINAL / NUM_FRAMES
num_samples = 4
NUM_STEPS = 100  # flow integration steps

SNAP_TIMES = [0.5, 1.2, 1.9, 2.7, 3.4, 3.9]
SNAP_FRAMES = [round(t / DT) for t in SNAP_TIMES]
CMAP, VMIN, VMAX, FIELD_LABEL = "RdBu_r", -1.5, 1.5, "activator u"
times = torch.arange(NUM_FRAMES, device=device) * DT

centres = (torch.arange(G, device=device) + 0.5) / G
YY, XX = torch.meshgrid(1.0 - centres, centres, indexing="ij")  # row 0 is y = 1
PACEMAKER_1 = (3 / G, 7 / G, 3 / G, 7 / G)  # lower-left, (x0, x1, y0, y1)
PACEMAKER_2 = (25 / G, 29 / G, 25 / G, 29 / G)  # upper-right
EARLY, LATE = 1.3, 1.8  # firing windows

## 2. The excitable-medium simulator

`FitzHughNagumoMap` is the generator $T_\vartheta$: explicit Euler with no-flux boundaries. Time is rescaled by `speed` so that a firing and its front fit in the window. The noise is strongest at the pacemakers, which is what makes their firing random.

In [ ]:
def box_mask(box):
    """(G, G) boolean mask of the pixels whose centres lie in a (x0, x1, y0, y1) box."""
    x0, x1, y0, y1 = box
    return (XX >= x0) & (XX <= x1) & (YY >= y0) & (YY <= y1)


class FitzHughNagumoMap(BaseMap):
    """Latent (B, 1, NUM_FRAMES * G * G) -> activator fields (B, 1, NUM_FRAMES * G * G)."""

    def __init__(self, drive=0.25, sigma=0.35, background_noise=0.3, diffusion=0.6, eps=0.08, a=0.7, b=0.8,
                 speed=3.0, substeps=4):
        super().__init__(latent_dim=NUM_FRAMES * G * G, output_shape=(NUM_FRAMES, G, G), device=device)
        pacemakers = (box_mask(PACEMAKER_1) | box_mask(PACEMAKER_2)).double()
        self.register_buffer("drive", drive * pacemakers)
        self.register_buffer("noise_scale", sigma * (pacemakers + background_noise))
        self.diffusion, self.eps, self.a, self.b = diffusion, eps, a, b
        self.speed, self.substeps = speed, substeps

    def _map(self, z):
        B = z.shape[0]
        z = z.reshape(B, NUM_FRAMES, G, G)
        u = -1.2 + 0.05 * z[:, 0]  # near the resting state (u, v) = (-1.2, -0.62)
        v = torch.full_like(u, -0.62)
        frames = [u]
        h = self.speed * DT / self.substeps
        for k in range(1, NUM_FRAMES):
            for _ in range(self.substeps):
                p = torch.nn.functional.pad(u[:, None], (1, 1, 1, 1), mode="replicate")[:, 0]  # no-flux
                laplacian = p[:, :-2, 1:-1] + p[:, 2:, 1:-1] + p[:, 1:-1, :-2] + p[:, 1:-1, 2:] - 4 * u
                u, v = (u + h * (self.diffusion * laplacian + u - u**3 / 3 - v + self.drive),
                        v + h * self.eps * (u + self.a - self.b * v))
            u = u + self.noise_scale * (self.speed * DT) ** 0.5 * z[:, k]
            frames.append(u)
        return torch.stack(frames, dim=1).reshape(B, 1, -1)


flow = GaussianIncrementFlow(map=FitzHughNagumoMap(), device=device)
sampler = IntegratorSampler(flow=flow)

## 3. Helpers

`sample` draws prior or guided fields, and `plot_snapshots` lays several regimes out as in the paper: two draws per regime, one row each, at six times.

In [ ]:
def sample(condition=None):
    """`num_samples` fields (B, num_frames, G, G): the prior if `condition` is None, else guided."""
    guidance = None if condition is None else GuidanceMC(flow, condition).to(device)
    out = sampler.sample(num_samples, guidance=guidance, num_steps=NUM_STEPS).detach()
    return out.reshape(num_samples, NUM_FRAMES, G, G)


def plot_snapshots(regimes, draws=2, overlay=None):
    """Rows: `draws` samples of each regime; columns: the fields at SNAP_TIMES.

    `regimes` is a list of (label, samples); `overlay(ax, label, frame)` draws a regime's
    condition on a panel.
    """
    rows = len(regimes) * draws
    fig, axes = plt.subplots(rows, len(SNAP_FRAMES), figsize=(1.7 * len(SNAP_FRAMES) + 1.6, 1.7 * rows),
                             squeeze=False)
    for r, (label, samples) in enumerate(regimes):
        for d in range(draws):
            row = r * draws + d
            for col, frame in enumerate(SNAP_FRAMES):
                ax = axes[row, col]
                im = ax.imshow(samples[d, frame].cpu(), cmap=CMAP, vmin=VMIN, vmax=VMAX)
                ax.set_xticks([])
                ax.set_yticks([])
                if overlay is not None:
                    overlay(ax, label, frame)
                if row == 0:
                    ax.set_title(f"t = {frame * DT:.2f}", fontsize=9)
            if d == 0:
                axes[row, 0].set_ylabel(label, fontsize=10, fontweight="bold")
    fig.colorbar(im, ax=axes, fraction=0.02, pad=0.01, label=FIELD_LABEL)
    return fig


def draw_box(ax, box, color):
    """Outline a (x0, x1, y0, y1) box of the unit square on an image panel (row 0 is y = 1)."""
    x0, x1, y0, y1 = box
    ax.add_patch(Rectangle((x0 * G - 0.5, (1 - y1) * G - 0.5), (x1 - x0) * G, (y1 - y0) * G,
                           fill=False, edgecolor=color, linewidth=1.6))


def smooth_max(values, tau):
    """Log-sum-exp maximum over the last axis: differentiable, within tau * log(n) of the hard max."""
    return tau * torch.logsumexp(values / tau, dim=-1)

## 4. Prior

Each pacemaker fires in roughly 40% of prior draws, at a random time, and some draws stay quiet.

In [ ]:
prior_samples = sample()
plot_snapshots([("Prior", prior_samples)], overlay=lambda ax, label, frame: [
    draw_box(ax, PACEMAKER_1, "white"), draw_box(ax, PACEMAKER_2, "white")]);

## 5. Firing conditions

`FiringPattern` is a `BinaryCondition` built from components `(pacemaker, t0, t1, fires)`. Each component takes the pacemaker-averaged activator, smooth-maximises it over the window $[t_0, t_1]$, and returns a margin that is positive when it crosses $0.5$ (fires) or stays below $0$ (silent).

In [ ]:
class FiringPattern(BinaryCondition):
    def __init__(self, components, v=1e-3, smooth_tau=0.1, fire_level=0.5, silent_level=0.0):
        super().__init__(v=v)
        self.components = [(box_mask(box).double(), t0, t1, fires) for box, t0, t1, fires in components]
        self.smooth_tau, self.fire_level, self.silent_level = smooth_tau, fire_level, silent_level

    def smooth_eval_condition(self, x):
        u = x.reshape(x.shape[0], NUM_FRAMES, G, G)
        margins = []
        for mask, t0, t1, fires in self.components:
            activity = (u * mask).sum(dim=(-1, -2)) / mask.sum()  # (B, NUM_FRAMES)
            window = (times >= t0) & (times <= t1)
            peak = smooth_max(activity[:, window], self.smooth_tau)
            margins.append(peak - self.fire_level if fires else self.silent_level - peak)
        return torch.stack(margins, dim=-1)[:, None, :]


def event_rate(condition, samples):
    margins = condition.smooth_eval_condition(samples.reshape(samples.shape[0], 1, -1)).squeeze(1)
    return (margins.min(dim=-1).values > 0).double().mean().item()


patterns = {
    "Source 1 only": FiringPattern([(PACEMAKER_1, 0.0, EARLY, True), (PACEMAKER_2, 0.0, T_FINAL, False)]),
    "Source 2 only": FiringPattern([(PACEMAKER_2, LATE, T_FINAL, True), (PACEMAKER_2, 0.0, LATE - DT, False),
                                    (PACEMAKER_1, 0.0, T_FINAL, False)]),
    "Both fire": FiringPattern([(PACEMAKER_1, 0.0, EARLY, True), (PACEMAKER_2, 0.0, LATE - DT, False),
                                (PACEMAKER_2, LATE, T_FINAL, True)]),
}
pattern_samples = {name: sample(condition) for name, condition in patterns.items()}

rate_samples = flow.map(torch.randn(400, 1, flow.latent_dim, device=device))  # prior batch for the rates
for name, condition in patterns.items():
    print(f"{name:14s} prior={event_rate(condition, rate_samples):.3f}  guided={event_rate(condition, pattern_samples[name]):.2f}")

## 6. Summary

The four regimes as in Figure 18 of the paper. Green boxes mark pacemakers required to fire, red boxes those required to stay silent; a box turns green or red only once its requirement applies.

In [ ]:
def overlay(ax, label, frame):
    t = frame * DT
    one = {"Source 1 only": "fire", "Source 2 only": "silent", "Both fire": "fire"}.get(label)
    two = {"Source 1 only": "silent", "Source 2 only": "fire" if t >= LATE else "silent",
           "Both fire": "fire" if t >= LATE else "silent"}.get(label)
    colors = {"fire": "lime", "silent": "red", None: "white"}
    draw_box(ax, PACEMAKER_1, colors[one])
    draw_box(ax, PACEMAKER_2, colors[two])


plot_snapshots([("Prior", prior_samples)] + list(pattern_samples.items()), overlay=overlay);